# Multimodal RAG Pipeline — Walkthrough

This notebook walks through every layer of the pipeline end-to-end:

| Step | What happens |
|------|--------------|
| 1 | **Text ingestion** — chunk → embed → store in ChromaDB |
| 2 | **Image ingestion** — CLIP encode → thumbnail → store |
| 3 | **Video ingestion** — keyframe extract → CLIP encode → store |
| 4 | **Hybrid retrieval** — BM25 + dense + RRF fusion + cross-encoder rerank |
| 5 | **Semantic cache** — cosine-distance lookup + store |
| 6 | **Answer generation** — Ollama llama3.2 with citation markers |
| 7 | **Observability** — Prometheus metrics + OpenTelemetry spans |
| 8 | **FastAPI** — live HTTP calls to the running service |

---

## Running mode

**Mock mode (default, no downloads):** All ML models are replaced with lightweight mocks — numpy arrays, deterministic embeddings. The pipeline logic, ChromaDB operations, and data flow are 100% real.

**Real mode:** Set `USE_REAL_MODELS = True` in the Setup cell. Requires:
```bash
# Models download automatically on first use (~500 MB total for text+CLIP)
pip install -r requirements.txt
```

> **Tip:** Run cells top-to-bottom. Each section builds on the previous one.

## 0. Setup

In [ ]:
import sys
import os

# Add project root to path so we can import pipeline modules
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# ── Toggle this to use real models ──────────────────────────────────────────
USE_REAL_MODELS = False
# ────────────────────────────────────────────────────────────────────────────

import tempfile
import numpy as np
import chromadb
from pipeline import get_collections

# Isolated in-memory ChromaDB for the demo
_tmpdir = tempfile.mkdtemp(prefix='rag_demo_')
chroma_client = chromadb.PersistentClient(path=_tmpdir)
text_col, image_col, video_col = get_collections(chroma_client)

print(f'ChromaDB ready at {_tmpdir}')
print(f'Collections: text_chunks={text_col.count()}, '
      f'image_embeddings={image_col.count()}, '
      f'video_keyframes={video_col.count()}')

In [ ]:
# ── Model loading ────────────────────────────────────────────────────────────
# Real models load from HuggingFace on first use.
# Mock models are lightweight numpy stubs with the same API surface.

import torch
from unittest.mock import MagicMock

if USE_REAL_MODELS:
    from pipeline.ingest import load_text_embedder, load_clip_model
    from pipeline.config import TEXT_EMBED_MODEL, CLIP_MODEL_ID
    from sentence_transformers import CrossEncoder
    from pipeline.config import RERANKER_MODEL

    print('Loading SentenceTransformer (all-MiniLM-L6-v2)...')
    text_embedder = load_text_embedder(TEXT_EMBED_MODEL)
    print('Loading CLIP (ViT-B/32)...')
    clip_model, clip_processor = load_clip_model(CLIP_MODEL_ID)
    print('Loading CrossEncoder (ms-marco-MiniLM-L6-v2)...')
    cross_encoder = CrossEncoder(RERANKER_MODEL)
    print('All real models loaded.')
else:
    # Mock text embedder: returns (n, 384) numpy array
    text_embedder = MagicMock()
    text_embedder.encode.side_effect = (
        lambda texts, **kw: np.random.RandomState(42).randn(len(texts), 384).astype(np.float32)
    )

    # Mock CLIP: returns unit vectors in 512-dim space
    clip_model = MagicMock()
    clip_processor = MagicMock()
    clip_model.get_image_features.return_value = torch.ones(1, 512)
    clip_model.get_text_features.return_value  = torch.ones(1, 512)
    clip_processor.return_value = {'pixel_values': torch.zeros(1, 3, 224, 224)}

    # Mock cross-encoder: returns a score per pair
    cross_encoder = MagicMock()
    cross_encoder.predict.side_effect = lambda pairs: [
        float(hash(p[1]) % 100) / 100 for p in pairs
    ]

    print('Mock models ready (set USE_REAL_MODELS=True for real embeddings)')

---
## 1. Text Ingestion

Text documents are split into overlapping chunks, embedded with SentenceTransformer, and upserted into the `text_chunks` ChromaDB collection.

```
Document  ──►  chunk_text()  ──►  [chunk_0, chunk_1, ...]  ──►  embed  ──►  ChromaDB
              (512 chars,              each chunk gets:
               50 overlap)              • id: {source_id}_chunk_{i}
                                        • embedding: 384-dim float32
                                        • metadata: {source_id, chunk_index, modality}
```

In [ ]:
from pipeline.ingest import chunk_text

rag_doc = """
Retrieval Augmented Generation (RAG) is a machine learning technique that enhances
large language model outputs by retrieving relevant documents at inference time.
Unlike purely parametric models that rely on training data alone, RAG systems
dynamically fetch external knowledge, reducing hallucination and enabling citation.

The retrieval step combines sparse methods (BM25) with dense vector search.
BM25 excels at exact keyword matching while dense retrieval captures semantic
similarity — even when the query uses different words than the document.
Reciprocal Rank Fusion (RRF) merges both ranked lists into a single ranking.

Cross-encoder reranking scores the top candidates jointly with the query,
providing higher precision at the cost of additional compute.
The reranked results are passed to an LLM (llama3.2 via Ollama) for answer generation.
"""

chunks = chunk_text(rag_doc, source_id='rag_overview.txt', collection=text_col,
                    embedder=text_embedder, chunk_size=256, overlap=30)

print(f'Created {len(chunks)} chunks from document')
print(f'ChromaDB text_chunks count: {text_col.count()}')
print()
for i, chunk in enumerate(chunks):
    print(f'  Chunk {i}: "{chunk[:80]}..."')

In [ ]:
# Ingest a few more documents to make retrieval interesting
docs = {
    'chromadb.txt':    'ChromaDB is an open-source vector database with cosine, L2, and '
                       'inner-product similarity. It supports EphemeralClient (in-memory) '
                       'and PersistentClient (disk-backed). Collections have immutable '
                       'hnsw:space metadata set at creation time.',
    'clip.txt':        'CLIP (Contrastive Language-Image Pretraining) by OpenAI encodes '
                       'images and text into a shared 512-dimensional embedding space. '
                       'The ViT-B/32 variant uses 32-pixel patches and enables zero-shot '
                       'image classification and cross-modal retrieval.',
    'observability.txt': 'Prometheus collects time-series metrics via pull-based scraping. '
                         'OpenTelemetry provides distributed tracing with context propagation '
                         'across service boundaries. Together they provide full observability '
                         'for multi-stage RAG pipelines.',
    'rrf.txt':         'Reciprocal Rank Fusion scores each document as the sum of 1/(k+rank) '
                       'across all ranked lists, where k=60 by default. Documents appearing '
                       'at the top of multiple lists receive higher scores. This is a '
                       'parameter-free method that outperforms linear combination in practice.',
}

for source_id, text in docs.items():
    chunks = chunk_text(text, source_id=source_id, collection=text_col, embedder=text_embedder)
    print(f'  {source_id}: {len(chunks)} chunk(s)')

print(f'\nTotal text_chunks: {text_col.count()}')

In [ ]:
# Inspect what's stored in ChromaDB
stored = text_col.get(include=['documents', 'metadatas'])

print(f'{'ID':<35} {'Source':<25} {'Chunk':<7} {'Text preview'}')
print('-' * 95)
for doc_id, doc, meta in zip(stored['ids'], stored['documents'], stored['metadatas']):
    preview = doc[:50].replace('\n', ' ') + '...'
    print(f'{doc_id:<35} {meta["source_id"]:<25} {meta["chunk_index"]:<7} {preview}')

---
## 2. Image Ingestion

Images are encoded with CLIP ViT-B/32 into 512-dim vectors. A 256×256 JPEG thumbnail is stored as base64 in the metadata for the Streamlit provenance UI.

```
Image file  ──►  CLIPProcessor  ──►  CLIPModel.get_image_features()  ──►  L2 normalise
                                                                               │
                                                               512-dim vector  │  + thumbnail_b64
                                                                               ▼
                                                               ChromaDB: image_embeddings
```

In [ ]:
import io
import tempfile
from PIL import Image as PILImage
from pipeline.ingest import embed_image

# Create three synthetic images with different colours for demo
image_specs = [
    ('sunset.jpg',   (255, 140,   0)),   # orange — sunset
    ('forest.jpg',   ( 34, 139,  34)),   # forest green
    ('ocean.jpg',    ( 30, 144, 255)),   # dodger blue
]

img_paths = []
for filename, colour in image_specs:
    img = PILImage.new('RGB', (128, 128), color=colour)
    tmp = tempfile.NamedTemporaryFile(suffix='.jpg', delete=False)
    img.save(tmp.name, format='JPEG')
    img_paths.append((tmp.name, filename))

for img_path, source_id in img_paths:
    doc_id = embed_image(img_path, source_id=source_id,
                         collection=image_col,
                         clip_model=clip_model,
                         clip_processor=clip_processor)
    print(f'  Embedded {source_id} → doc_id={doc_id}')

print(f'\nTotal image_embeddings: {image_col.count()}')

In [ ]:
# Inspect the stored image metadata — thumbnail is base64 JPEG
import base64

stored_imgs = image_col.get(include=['metadatas'])
print(f"{'Doc ID':<30} {'Source':<20} {'Thumbnail size (chars)'}")
print('-' * 65)
for doc_id, meta in zip(stored_imgs['ids'], stored_imgs['metadatas']):
    thumb_size = len(meta.get('thumbnail_b64', ''))
    print(f"{doc_id:<30} {meta['source_id']:<20} {thumb_size}")

# Decode and display the first thumbnail
b64 = stored_imgs['metadatas'][0].get('thumbnail_b64', '')
if b64:
    from IPython.display import display
    thumb = PILImage.open(io.BytesIO(base64.b64decode(b64)))
    print(f'\nThumbnail size: {thumb.size}')
    display(thumb)

---
## 3. Video Ingestion

Videos are sampled every N seconds with OpenCV. Each keyframe is CLIP-embedded and stored alongside its timestamp and a base64 thumbnail.

```
Video file  ──►  OpenCV VideoCapture  ──►  sample frame every 5s
                                                  │
                                         CLIP embed + thumbnail
                                                  │
                                         ChromaDB: video_keyframes
                                         id: vid_{source_id}_frame_{i}
                                         metadata: {timestamp_sec, thumbnail_b64}
```

In [ ]:
import cv2
from pipeline.ingest import extract_keyframes

# Synthesise a short 10-second video (3 fps, 30 frames total)
vid_path = tempfile.NamedTemporaryFile(suffix='.avi', delete=False).name
fourcc = cv2.VideoWriter_fourcc(*'MJPG')
fps, width, height = 3, 128, 128
writer = cv2.VideoWriter(vid_path, fourcc, fps, (width, height))

colours = [
    (200, 100,  50),  # first 5s  — warm tones
    ( 50, 150, 200),  # middle 5s — cool tones
    (100, 200,  50),  # last 5s   — green tones
]
for i in range(30):
    r, g, b = colours[i // 10]
    frame = np.full((height, width, 3), [b, g, r], dtype=np.uint8)  # BGR
    writer.write(frame)
writer.release()
print(f'Synthesised video: {vid_path} ({fps} fps, {30 // fps}s)')

# Extract keyframes every 3 seconds
frame_ids = extract_keyframes(
    vid_path, source_id='demo_video.avi',
    collection=video_col,
    clip_model=clip_model,
    clip_processor=clip_processor,
    sample_interval_sec=3,
)
print(f'\nExtracted {len(frame_ids)} keyframes:')
for fid in frame_ids:
    print(f'  {fid}')
print(f'\nTotal video_keyframes: {video_col.count()}')

In [ ]:
# Show keyframe thumbnails and timestamps
from IPython.display import display

stored_vid = video_col.get(include=['metadatas'])
print(f"{'Frame ID':<35} {'Timestamp (s)'}")
print('-' * 50)
for doc_id, meta in zip(stored_vid['ids'], stored_vid['metadatas']):
    ts = meta.get('timestamp_sec', '?')
    print(f"{doc_id:<35} {ts}s")

print('\nKeyframe thumbnails:')
for meta in stored_vid['metadatas']:
    b64 = meta.get('thumbnail_b64', '')
    if b64:
        thumb = PILImage.open(io.BytesIO(base64.b64decode(b64)))
        print(f"  t={meta.get('timestamp_sec')}s → {thumb.size}")
        display(thumb)

---
## 4. Hybrid Retrieval

Four signals are fused for every query:

```
Query text
    │
    ├──► BM25 (text_chunks)          ──► ranked list 1
    ├──► Dense text (text_chunks)    ──► ranked list 2   ──►  RRF fusion
    ├──► CLIP text→image             ──► ranked list 3   ──►  (k=60)
    └──► CLIP text→video             ──► ranked list 4   ──►
                                                               │
                                                   Cross-encoder rerank
                                                   (text candidates only)
                                                               │
                                                        Final top-k results
```

In [ ]:
# ── Step-by-step: BM25 retrieval ─────────────────────────────────────────────
from pipeline.retrieve import BM25Index

bm25 = BM25Index(text_col)
print(f'BM25 index built from {len(bm25.docs)} documents')

query = 'what is retrieval augmented generation?'
bm25_results = bm25.query(query, top_k=5)

print(f'\nBM25 results for: "{query}"')
print(f"{'Rank':<6} {'Score':<10} {'Source':<25} {'Text preview'}")
print('-' * 80)
for r in bm25_results:
    preview = r['document'][:50].replace('\n', ' ') + '...'
    print(f"{r['rank']:<6} {r['score']:<10.4f} {r['metadata']['source_id']:<25} {preview}")

In [ ]:
# ── Step-by-step: Dense vector retrieval ─────────────────────────────────────
from pipeline.retrieve import dense_query

# Embed the query with SentenceTransformer
query_embedding = text_embedder.encode([query], show_progress_bar=False)[0].tolist()
print(f'Query embedding: {len(query_embedding)}-dim vector')
print(f'First 8 values: {[round(v, 4) for v in query_embedding[:8]]}...')

dense_results = dense_query(text_col, query_embedding, top_k=5)

print(f'\nDense retrieval results:')
print(f"{'Rank':<6} {'Cosine sim':<12} {'Source':<25} {'Text preview'}")
print('-' * 80)
for r in dense_results:
    preview = r['document'][:50].replace('\n', ' ') + '...'
    print(f"{r['rank']:<6} {r['score']:<12.4f} {r['metadata']['source_id']:<25} {preview}")

In [ ]:
# ── Step-by-step: RRF fusion ──────────────────────────────────────────────────
from pipeline.retrieve import rrf_fusion

# Also get CLIP results for images and video
from pipeline.ingest import clip_embed_text
clip_query_emb = clip_embed_text(query, clip_model, clip_processor)

dense_image_results = dense_query(image_col, clip_query_emb, top_k=5)
dense_video_results = dense_query(video_col, clip_query_emb, top_k=5)

print(f'BM25: {len(bm25_results)} results')
print(f'Dense text: {len(dense_results)} results')
print(f'Dense image: {len(dense_image_results)} results')
print(f'Dense video: {len(dense_video_results)} results')

fused = rrf_fusion([bm25_results, dense_results, dense_image_results, dense_video_results])

print(f'\nRRF fused ranking ({len(fused)} unique documents):')
print(f"{'Rank':<6} {'RRF score':<12} {'Modality':<10} {'Source'}")
print('-' * 55)
for i, r in enumerate(fused[:10], 1):
    modality = r.get('metadata', {}).get('modality', '?')
    source   = r.get('metadata', {}).get('source_id', '?')
    icon = {'text': '📄', 'image': '🖼️', 'video': '🎬'}.get(modality, '❓')
    print(f"{i:<6} {r['rrf_score']:<12.6f} {icon} {modality:<8} {source}")

In [ ]:
# ── Step-by-step: Cross-encoder reranking ─────────────────────────────────────
from pipeline.retrieve import rerank

reranked = rerank(query, fused, cross_encoder, top_k=5)

print(f'Top-5 after cross-encoder reranking:')
print(f"{'Rank':<6} {'RRF score':<12} {'Rerank score':<14} {'Modality':<10} {'Source'}")
print('-' * 70)
for i, r in enumerate(reranked, 1):
    modality     = r.get('metadata', {}).get('modality', '?')
    source       = r.get('metadata', {}).get('source_id', '?')
    rrf_score    = r.get('rrf_score', 0)
    rerank_score = r.get('rerank_score')
    rerank_str   = f'{rerank_score:.4f}' if rerank_score is not None else 'N/A (visual)'
    icon = {'text': '📄', 'image': '🖼️', 'video': '🎬'}.get(modality, '❓')
    print(f"{i:<6} {rrf_score:<12.6f} {rerank_str:<14} {icon} {modality:<8} {source}")

In [ ]:
# ── Full pipeline convenience function ───────────────────────────────────────
from pipeline.retrieve import hybrid_retrieve

results = hybrid_retrieve(
    query_text=query,
    text_collection=text_col,
    image_collection=image_col,
    video_collection=video_col,
    text_embedder=text_embedder,
    clip_model=clip_model,
    clip_processor=clip_processor,
    cross_encoder=cross_encoder,
    top_k=5,
)

print(f'hybrid_retrieve returned {len(results)} results for: "{query}"')
for i, r in enumerate(results, 1):
    m = r.get('metadata', {})
    icon = {'text': '📄', 'image': '🖼️', 'video': '🎬'}.get(m.get('modality'), '❓')
    print(f'  {i}. {icon} {m.get("source_id","?")}  RRF={r["rrf_score"]:.6f}')

---
## 5. Answer Generation

The retrieved results are formatted into a prompt with `[N]` citation markers, then sent to Ollama.

```
retrieval_results
    │
    ▼
build_prompt(query, results)   ←── [1] source_id (modality): text content...
    │                               [2] source_id (modality): text content...
    ▼                               [visual content] for images/video
OllamaClient.generate_text()   ←── llama3.2 (text-only results)
      or
OllamaClient.generate_vision() ←── llama3.2-vision (image in top-3)
    │
    ▼
{ answer, citations, retrieval_results }
```

In [ ]:
from pipeline.generate import build_prompt

prompt = build_prompt(query, results)
print('Built prompt:')
print('=' * 70)
print(prompt)
print('=' * 70)

In [ ]:
import asyncio
from unittest.mock import AsyncMock
from pipeline.generate import OllamaClient, generate_answer

# Mock Ollama so the notebook runs without a running Ollama server.
# Set USE_REAL_OLLAMA = True and start `ollama serve` + pull llama3.2 to use the real LLM.
USE_REAL_OLLAMA = False

if USE_REAL_OLLAMA:
    ollama_client = OllamaClient()
else:
    ollama_client = MagicMock()
    ollama_client.generate_text = AsyncMock(
        return_value=(
            'Retrieval Augmented Generation (RAG) [1] is a technique that enhances '
            'language model outputs by fetching relevant documents at inference time [2]. '
            'It combines BM25 sparse retrieval with dense vector search [3], '
            'fused via Reciprocal Rank Fusion (RRF) [4]. '
            'Cross-encoder reranking further improves precision before the final answer is generated.'
        )
    )
    ollama_client.generate_vision = AsyncMock(
        return_value='The visual content shows relevant imagery related to the query.'
    )

result = await generate_answer(
    query=query,
    retrieval_results=results,
    ollama_client=ollama_client,
)

print('Answer:')
print('-' * 60)
print(result['answer'])
print()
print(f'Citations: {result["citations"]}')

---
## 6. Semantic Cache

The cache stores query results in SQLite using cosine distance between embeddings.

- **Hit threshold:** cosine distance < `0.05`  (= cosine similarity > 0.95)
- **Miss:** run full retrieval + generation pipeline
- **Why cosine *distance*?** ChromaDB returns distances; `1 - similarity = distance`, so distance < 0.05 means >95% similarity.

```
lookup(query)
  ├── embed(query) → 384-dim vector
  ├── load all cached embeddings from SQLite
  ├── compute cosine_distance(query_emb, each_cached_emb)
  └── if min_distance < 0.05 → return cached result  (⚡ cache hit)
                             → return None           (🔄 cache miss)
```

In [ ]:
from pipeline.cache import SemanticCache

# In-memory SQLite cache (use a file path for persistence)
cache = SemanticCache(db_path=':memory:', embedder=text_embedder)

# ── Miss ──────────────────────────────────────────────────────────────────────
first_lookup = cache.lookup(query)
print(f'Before storing — cache hit: {first_lookup is not None}')

# ── Store ─────────────────────────────────────────────────────────────────────
cache.store(query, result)
print(f'Stored result for: "{query}"')
print(f'Cache size: {cache.count()} entries')

# ── Exact hit ─────────────────────────────────────────────────────────────────
hit = cache.lookup(query)
if hit:
    print(f'\n⚡ Cache hit! distance={hit.get("cache_distance", "?"):.6f}')
    print(f'   Matched query: "{hit.get("matched_query", "?")}"')
    print(f'   Answer (first 80 chars): "{hit["answer"][:80]}..."')
else:
    print('Cache miss (expected hit — embedder may be stochastic in mock mode)')

In [ ]:
# Simulate near-duplicate queries (same meaning, different phrasing)
near_duplicates = [
    'what is RAG in machine learning?',
    'explain retrieval augmented generation',
    'how does RAG work?',
]
different_query = 'what is the capital of France?'

print('Near-duplicate queries (may hit or miss depending on embedder):')
for q in near_duplicates:
    hit = cache.lookup(q)
    status = f'⚡ HIT  dist={hit["cache_distance"]:.4f}' if hit else '🔄 MISS'
    print(f'  {status}  "{q}"')

print()
hit = cache.lookup(different_query)
status = f'⚡ HIT  dist={hit["cache_distance"]:.4f}' if hit else '🔄 MISS'
print(f'Different query:  {status}  "{different_query}"')

---
## 7. Observability

### 7a. Prometheus Metrics

All pipeline events are tracked with a custom `CollectorRegistry` (never the global default — that causes duplicate timeseries errors in tests).

In [ ]:
from observability.metrics import (
    CACHE_HITS, CACHE_MISSES, QUERY_COUNT,
    INGEST_COUNT, RETRIEVAL_LATENCY, GENERATION_LATENCY,
    get_metrics_summary, get_prometheus_output,
)
import time

# Simulate some pipeline events
QUERY_COUNT.labels(status='success').inc(3)
QUERY_COUNT.labels(status='cache_hit').inc(2)
CACHE_HITS.inc(2)
CACHE_MISSES.inc(3)
INGEST_COUNT.labels(modality='text').inc(5)
INGEST_COUNT.labels(modality='image').inc(3)
INGEST_COUNT.labels(modality='video').inc(1)

with RETRIEVAL_LATENCY.time():
    time.sleep(0.015)  # simulate 15ms retrieval

GENERATION_LATENCY.observe(2.3)  # simulate 2.3s generation

# Summary dict used by the /stats API endpoint
summary = get_metrics_summary()
print('Metrics summary (from /stats endpoint):')
for k, v in summary.items():
    val = f'{v:.1%}' if 'rate' in k else f'{v:.1f}' if isinstance(v, float) else v
    print(f'  {k:<35} {val}')

In [ ]:
# Raw Prometheus text format (what GET /metrics returns)
output_bytes, content_type = get_prometheus_output()
output_text = output_bytes.decode('utf-8')

print(f'Content-Type: {content_type}')
print(f'Output size: {len(output_bytes)} bytes')
print('\n--- Prometheus text format (rag_ metrics only) ---')
for line in output_text.splitlines():
    if line.startswith('rag_') or line.startswith('# HELP rag_') or line.startswith('# TYPE rag_'):
        print(line)

### 7b. OpenTelemetry Tracing

Each pipeline stage is wrapped with `trace_stage()` — a context manager that creates a named span with custom attributes.

In [ ]:
from observability.tracing import trace_stage, get_finished_spans, clear_spans

clear_spans()

# Simulate a full query pipeline with tracing
with trace_stage('cache_lookup', {'query': query[:80]}):
    time.sleep(0.002)  # 2ms cache lookup

with trace_stage('retrieval', {'query': query[:80], 'top_k': 5}):
    time.sleep(0.015)  # 15ms retrieval

with trace_stage('reranking', {'candidates': 15}):
    time.sleep(0.008)  # 8ms reranking

with trace_stage('generation', {'model': 'llama3.2', 'query': query[:80]}):
    time.sleep(0.001)  # 1ms (mocked)

spans = get_finished_spans()
print(f'Recorded {len(spans)} spans:\n')
print(f"{'Stage':<20} {'Duration (ms)':<16} {'Attributes'}")
print('-' * 70)
for span in spans:
    duration_ms = (span.end_time - span.start_time) / 1_000_000
    attrs = dict(span.attributes or {})
    print(f"{span.name:<20} {duration_ms:<16.1f} {attrs}")

---
## 8. FastAPI Endpoints

The full pipeline is served via FastAPI. All shared resources are initialised once in the `lifespan` context manager and stored in `app.state`.

Here we use `httpx.ASGITransport` to call the app directly — **no live server needed**.

In [ ]:
import httpx
from api.app import app
from pipeline.cache import SemanticCache

# Inject our demo collections and mock models into app.state
app.state.chroma_client  = chroma_client
app.state.text_col       = text_col
app.state.image_col      = image_col
app.state.video_col      = video_col
app.state.text_embedder  = text_embedder
app.state.clip_model     = clip_model
app.state.clip_processor = clip_processor
app.state.cross_encoder  = cross_encoder
app.state.ollama_client  = ollama_client
app.state.cache = SemanticCache(db_path=':memory:', embedder=text_embedder)

transport = httpx.ASGITransport(app=app)

async with httpx.AsyncClient(transport=transport, base_url='http://demo') as client:
    # GET /health
    resp = await client.get('/health')
    print('GET /health')
    print(f'  Status: {resp.status_code}')
    import json
    print(f'  Body:   {json.dumps(resp.json(), indent=4)}')

In [ ]:
async with httpx.AsyncClient(transport=transport, base_url='http://demo') as client:
    # GET /stats
    resp = await client.get('/stats')
    print('GET /stats')
    print(f'  Status: {resp.status_code}')
    for k, v in resp.json().items():
        print(f'  {k:<35} {v}')

In [ ]:
import io as _io

async with httpx.AsyncClient(transport=transport, base_url='http://demo', timeout=30.0) as client:
    # POST /ingest — text file
    content = b'Semantic search uses neural embeddings to find similar documents by meaning. ' * 10
    resp = await client.post(
        '/ingest',
        files={'file': ('semantic_search.txt', _io.BytesIO(content), 'text/plain')},
        data={'modality': 'text'},
    )
    print('POST /ingest (text file)')
    print(f'  Status: {resp.status_code}')
    print(f'  Body:   {json.dumps(resp.json(), indent=4)}')

In [ ]:
async with httpx.AsyncClient(transport=transport, base_url='http://demo', timeout=60.0) as client:
    # POST /query — first call (cache miss)
    resp = await client.post('/query', json={
        'query': 'how does retrieval augmented generation work?',
        'top_k': 3,
        'use_cache': True,
    })
    data = resp.json()
    print('POST /query — first call')
    print(f'  Status:     {resp.status_code}')
    print(f'  Cache hit:  {data["cache_hit"]} ({"⚡" if data["cache_hit"] else "🔄"})')
    print(f'  Latency:    {data["latency_ms"]:.1f} ms')
    print(f'  Citations:  {data["citations"]}')
    print(f'  Answer:     {data["answer"][:120]}...')
    print()
    print(f'  Top retrieval results:')
    for r in data['retrieval_results']:
        m = r.get('metadata', {})
        icon = {'text': '📄', 'image': '🖼️', 'video': '🎬'}.get(m.get('modality'), '❓')
        print(f"    {icon} {m.get('source_id','?'):<25} RRF={r.get('rrf_score',0):.6f}")

In [ ]:
async with httpx.AsyncClient(transport=transport, base_url='http://demo', timeout=60.0) as client:
    # POST /query — second call (should be cache hit with real embedder, miss with mock)
    resp = await client.post('/query', json={
        'query': 'how does retrieval augmented generation work?',
        'top_k': 3,
        'use_cache': True,
    })
    data = resp.json()
    status = '⚡ Cache hit' if data['cache_hit'] else '🔄 Cache miss'
    print(f'POST /query — second call (same query)')
    print(f'  {status}')
    print(f'  Latency: {data["latency_ms"]:.1f} ms')
    print()
    if data['cache_hit']:
        print('  Result served from cache — Ollama was NOT called.')
    else:
        print('  Note: mock embedder uses random state so identical queries'
              ' may produce different embeddings. Set USE_REAL_MODELS=True for cache hits.')

---
## 9. What the Streamlit Dashboard looks like

The dashboard at `dashboard/app.py` calls the FastAPI service over HTTP and renders:

- **⚡ Cache hit** or **🔄 Cache miss** badge
- Answer with formatted markdown
- **Retrieval Provenance** — one row per result:
  - Modality icon (📄/🖼️/🎬) + source filename
  - RRF score, dense score, rerank score as `st.metric` widgets
  - Progress bar scaled to RRF contribution
  - Inline thumbnail via `st.image` for images and video frames
- **Score breakdown** bar chart of all sources' RRF contributions
- Raw JSON expander

To run it:
```bash
# Terminal 1
ollama serve

# Terminal 2
uvicorn api.app:app --reload --port 8000

# Terminal 3
streamlit run dashboard/app.py
# Open http://localhost:8501
```

The architecture intentionally keeps the dashboard **stateless** — it never imports pipeline modules directly. It only makes HTTP calls to the FastAPI service, so the two components can be deployed independently.

In [ ]:
# Simulate what the dashboard renders for a query response
from IPython.display import HTML, display

async with httpx.AsyncClient(transport=transport, base_url='http://demo', timeout=60.0) as client:
    resp = await client.post('/query', json={
        'query': 'what is CLIP and how is it used?',
        'top_k': 5,
        'use_cache': False,
    })
data = resp.json()

cache_badge = '⚡ Cache hit' if data['cache_hit'] else '🔄 Cache miss — full pipeline'
citations   = ' · '.join(f'`{c}`' for c in set(data['citations']))

rows = []
for i, r in enumerate(data['retrieval_results']):
    m = r.get('metadata', {})
    icon     = {'text': '📄', 'image': '🖼️', 'video': '🎬'}.get(m.get('modality'), '❓')
    source   = m.get('source_id', '?')
    rrf      = r.get('rrf_score', 0)
    dense    = r.get('score', 0)
    rerank   = r.get('rerank_score')
    bar_pct  = min(int(rrf * 200 * 100), 100)
    rerank_s = f'{rerank:.3f}' if rerank is not None else 'N/A'
    rows.append(f"""
        <tr>
          <td style='font-size:1.4em'>{icon}</td>
          <td><b>{source}</b></td>
          <td>{rrf:.6f}</td>
          <td>{dense:.3f}</td>
          <td>{rerank_s}</td>
          <td><div style='background:#4CAF50;height:12px;width:{bar_pct}%;border-radius:4px'></div></td>
        </tr>""")

html = f"""
<div style='font-family:sans-serif;max-width:800px'>
  <div style='background:{'#d4edda' if data['cache_hit'] else '#cce5ff'};padding:8px 12px;
              border-radius:6px;margin-bottom:12px'>{cache_badge} &nbsp;|
              ⏱ {data['latency_ms']:.0f} ms</div>
  <h3>💬 Answer</h3>
  <p>{data['answer']}</p>
  <p style='color:#555'>📚 Sources: {citations or 'none'}</p>
  <hr/>
  <h3>🔎 Retrieval Provenance ({len(data['retrieval_results'])} results)</h3>
  <table style='border-collapse:collapse;width:100%'>
    <tr style='background:#f0f2f6'>
      <th></th><th>Source</th><th>RRF Score</th>
      <th>Dense</th><th>Rerank</th><th>Contribution</th>
    </tr>
    {''.join(rows)}
  </table>
</div>"""

display(HTML(html))

---
## Summary

You've walked through the full Multimodal RAG Pipeline:

| Component | File | Key function/class |
|-----------|------|-------------------|
| Text ingestion | `pipeline/ingest.py` | `chunk_text()` |
| Image ingestion | `pipeline/ingest.py` | `embed_image()` |
| Video ingestion | `pipeline/ingest.py` | `extract_keyframes()` |
| BM25 retrieval | `pipeline/retrieve.py` | `BM25Index` |
| Dense retrieval | `pipeline/retrieve.py` | `dense_query()` |
| RRF fusion | `pipeline/retrieve.py` | `rrf_fusion()` |
| Reranking | `pipeline/retrieve.py` | `rerank()` |
| Full retrieval | `pipeline/retrieve.py` | `hybrid_retrieve()` |
| Prompt building | `pipeline/generate.py` | `build_prompt()` |
| Answer generation | `pipeline/generate.py` | `generate_answer()` |
| Semantic cache | `pipeline/cache.py` | `SemanticCache` |
| Prometheus metrics | `observability/metrics.py` | `get_metrics_summary()` |
| OTel tracing | `observability/tracing.py` | `trace_stage()` |
| REST API | `api/app.py` | FastAPI app |
| Dashboard | `dashboard/app.py` | Streamlit app |

### Next steps

```bash
# Run the full stack
python cli/download_demo.py          # download demo data
python cli/ingest_cli.py data/       # ingest into ChromaDB
uvicorn api.app:app --port 8000      # start API server
streamlit run dashboard/app.py       # start dashboard

# Run all tests (no external services needed)
pytest tests/ -v -m "not slow"
```